# Single-Block Baseline Testing

This notebook compares increasingly expressive replacement families for zero-based SmolLM2 layer 11. Parameter budget is treated as a separate axis from function-class complexity. All candidates use the same dense-model activation pairs and are inserted one at a time without recovery.

## Complexity order

zero -> mean -> low-rank linear -> dense linear -> standard MLP -> gated MLP -> linear plus nonlinear residual

Every section appends to the same result table. The final question is whether an added architectural feature improves the parameter-quality frontier.

| Tier | Approximate cost | Linear | Standard MLP | Gated MLP | Hybrid (r, m) |
|---|---:|---:|---:|---:|---:|
| B1 | 0.52M / 1.04% | rank 128 | width 128 | width 85 | 64, 64 |
| B2 | 1.05M / 2.08% | rank 256 | width 256 | width 171 | 128, 128 |
| B3 | 2.10M / 4.17% | rank 512 | width 512 | width 341 | 256, 256 |
| B4 | 4.19M / 8.33% | dense | width 1024 | width 683 | 512, 512 |

In [ ]:
from dataclasses import asdict
from pathlib import Path
import json
import sys

def find_project_root(start):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'mlp_replacement').is_dir():
            return candidate
    raise RuntimeError('Could not locate the repository root')

PROJECT_ROOT = find_project_root(Path.cwd())
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch

from mlp_replacement.capture import ActivationPairs, collect_module_io
from mlp_replacement.config import DataConfig, ModelConfig, OperatorConfig
from mlp_replacement.data import build_data_loaders
from mlp_replacement.evaluation.language_model import evaluate_language_model
from mlp_replacement.evaluation.operator import evaluate_operator
from mlp_replacement.model import get_mlp_block, load_model_and_tokenizer
from mlp_replacement.operators import (
    BottleneckMLPReplacement, GatedMLPReplacement, HybridReplacement,
    LowRankLinearReplacement, MeanReplacement, ZeroReplacement,
    fit_operator, fit_ridge_linear, initialize_low_rank_from_svd, linear_svd,
)
from mlp_replacement.runlog import environment_record, json_value
from mlp_replacement.surgery import (
    count_parameters, count_state_elements, temporary_replacement,
)

sns.set_theme(style='whitegrid', context='notebook')

In [ ]:
TARGET_LAYER = 11
SEED = 21
RIDGE = 1e-4
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'results' / 'notebook-block-study' / 'baselines-layer-11.json'

BUDGETS = (
    {'tier': 'B1', 'linear_rank': 128, 'standard_width': 128, 'gated_width': 85, 'hybrid_rank': 64, 'hybrid_width': 64},
    {'tier': 'B2', 'linear_rank': 256, 'standard_width': 256, 'gated_width': 171, 'hybrid_rank': 128, 'hybrid_width': 128},
    {'tier': 'B3', 'linear_rank': 512, 'standard_width': 512, 'gated_width': 341, 'hybrid_rank': 256, 'hybrid_width': 256},
    {'tier': 'B4', 'linear_rank': None, 'standard_width': 1024, 'gated_width': 683, 'hybrid_rank': 512, 'hybrid_width': 512},
)
model_config = ModelConfig(model_id='HuggingFaceTB/SmolLM2-1.7B', device='auto', dtype='auto')
data_config = DataConfig(
    sequence_length=128, batch_size=2,
    num_calibration_batches=48, num_operator_validation_batches=24,
    num_recovery_batches=0, num_recovery_validation_batches=0,
    num_model_validation_batches=24, num_test_batches=0, seed=SEED,
)
training_config = OperatorConfig(
    kind='linear', epochs=20, learning_rate=1e-3, batch_size=2048,
    weight_decay=0.0, scheduler='constant', early_stopping_patience=3, seed=SEED,
)
torch.manual_seed(SEED)

In [ ]:
model, tokenizer = load_model_and_tokenizer(model_config)
device = next(model.parameters()).device
loaders = build_data_loaders(tokenizer, data_config, include_recovery=False)
block = get_mlp_block(model, TARGET_LAYER)
capture_kwargs = dict(device=device, storage_device='cpu', storage_dtype=torch.float32)
training_pairs = collect_module_io(
    model, block.path, loaders.calibration, data_config.num_calibration_batches, **capture_kwargs
)
validation_pairs = collect_module_io(
    model, block.path, loaders.operator_validation, data_config.num_operator_validation_batches, **capture_kwargs
)
hidden_size = training_pairs.hidden_size
original_parameters = count_parameters(block.module)
print({'device': str(device), 'hidden_size': hidden_size, 'original_parameters': original_parameters})

In [ ]:
baseline_lm = evaluate_language_model(
    model, loaders.model_validation, device, data_config.num_model_validation_batches
)
results = [{
    'name': 'original_mlp', 'family': 'teacher', 'tier': 'teacher',
    'parameters': original_parameters, 'state_elements': original_parameters,
    'cost_elements': original_parameters,
    'relative_block_parameters': 1.0, 'removed_parameters': 0,
    'local_mse': 0.0, 'local_relative_mse': 0.0, 'local_r2': 1.0,
    'local_cosine': 1.0, 'local_norm_ratio': 1.0,
    'median_token_relative_error': 0.0, 'p95_token_relative_error': 0.0,
    'loss': baseline_lm.loss, 'perplexity': baseline_lm.perplexity,
    'delta_loss': 0.0, 'delta_perplexity': 0.0, 'recovery': False,
}]
training_histories = {}
baseline_lm

In [ ]:
def append_candidate(name, family, tier, module, history=()):
    module = module.to(device)
    local = evaluate_operator(module, validation_pairs, device, training_config.batch_size)
    with temporary_replacement(model, TARGET_LAYER, module) as record:
        integrated = evaluate_language_model(
            model, loaders.model_validation, device, data_config.num_model_validation_batches
        )
    parameters = count_parameters(module)
    state_elements = count_state_elements(module)
    results.append({
        'name': name, 'family': family, 'tier': tier,
        'parameters': parameters, 'state_elements': state_elements,
        'cost_elements': state_elements,
        'relative_block_parameters': parameters / original_parameters,
        'removed_parameters': record.original_parameters - parameters,
        'local_mse': local.mse, 'local_relative_mse': local.relative_mse,
        'local_r2': local.r2, 'local_cosine': local.cosine_similarity,
        'local_norm_ratio': local.norm_ratio,
        'median_token_relative_error': local.median_token_relative_error,
        'p95_token_relative_error': local.p95_token_relative_error,
        'loss': integrated.loss, 'perplexity': integrated.perplexity,
        'delta_loss': integrated.loss - baseline_lm.loss,
        'delta_perplexity': integrated.perplexity - baseline_lm.perplexity,
        'recovery': False,
    })
    training_histories[name] = [asdict(epoch) for epoch in history]
    return results[-1]

def predict_cpu(module, inputs, batch_size=2048):
    module = module.to(device)
    dtype = next(module.parameters()).dtype
    chunks = []
    module.eval()
    with torch.no_grad():
        for start in range(0, inputs.shape[0], batch_size):
            chunks.append(module(inputs[start:start + batch_size].to(device, dtype=dtype)).float().cpu())
    return torch.cat(chunks)

## 1. Trivial controls

Zero output measures the effect of skipping the MLP contribution. Mean output tests whether an input-independent approximation captures meaningful behavior. The mean is stored as a buffer, so parameter and stored-state counts are reported separately.

In [ ]:
append_candidate('zero', 'constant', 'control', ZeroReplacement())
append_candidate('mean', 'constant', 'control', MeanReplacement(training_pairs.targets.mean(dim=0)))
pd.DataFrame(results).tail(2)

## 2. Linear family

Fit one regularized dense linear map, compute its SVD once, initialize all rank-constrained factors from that decomposition, and refine them against the actual activation MSE. The dense map is the linear-function ceiling at B4.

In [ ]:
dense_linear = fit_ridge_linear(training_pairs, ridge=RIDGE, bias=False, device=device).to(device)
dense_svd = tuple(value.cpu() for value in linear_svd(dense_linear))
for budget in BUDGETS[:3]:
    torch.manual_seed(SEED)
    module = LowRankLinearReplacement(hidden_size, budget['linear_rank'], bias=False)
    initialize_low_rank_from_svd(module, *dense_svd)
    fit = fit_operator(module, training_pairs, validation_pairs, training_config, device)
    append_candidate(
        f"linear_rank_{budget['linear_rank']}", 'low_rank_linear', budget['tier'],
        fit.module, fit.history,
    )
append_candidate('dense_linear', 'dense_linear', 'B4', dense_linear)
linear_rows = pd.DataFrame(results)
linear_rows[linear_rows['family'].str.contains('linear')]

## 3. Compact nonlinear families

The standard MLP adds nonlinearity without gating. The narrow gated MLP preserves the teacher's SwiGLU-style structure. Both are evaluated at approximately matched instantiated parameter counts.

In [ ]:
for budget in BUDGETS:
    width = budget['standard_width']
    torch.manual_seed(SEED)
    module = BottleneckMLPReplacement(hidden_size, width / hidden_size, activation='silu', bias=False)
    fit = fit_operator(module, training_pairs, validation_pairs, training_config, device)
    append_candidate(f'standard_mlp_{width}', 'standard_mlp', budget['tier'], fit.module, fit.history)
pd.DataFrame(results).query("family == 'standard_mlp'")

In [ ]:
for budget in BUDGETS:
    width = budget['gated_width']
    torch.manual_seed(SEED)
    module = GatedMLPReplacement(hidden_size, width, bias=False)
    fit = fit_operator(module, training_pairs, validation_pairs, training_config, device)
    append_candidate(f'gated_mlp_{width}', 'gated_mlp', budget['tier'], fit.module, fit.history)
pd.DataFrame(results).query("family == 'gated_mlp'")

## 4. Linear plus nonlinear residual

The linear branch receives the dense-map SVD initialization. The nonlinear branch first learns the residual target y minus the linear prediction. Both branches are then jointly fine-tuned on y.

In [ ]:
for budget in BUDGETS:
    rank = budget['hybrid_rank']
    width = budget['hybrid_width']
    torch.manual_seed(SEED)
    hybrid = HybridReplacement(hidden_size, rank, width, activation='silu', bias=False)
    initialize_low_rank_from_svd(hybrid.linear, *dense_svd)
    train_linear = predict_cpu(hybrid.linear, training_pairs.inputs)
    validation_linear = predict_cpu(hybrid.linear, validation_pairs.inputs)
    residual_training = ActivationPairs(training_pairs.inputs, training_pairs.targets - train_linear)
    residual_validation = ActivationPairs(validation_pairs.inputs, validation_pairs.targets - validation_linear)
    residual_fit = fit_operator(hybrid.nonlinear, residual_training, residual_validation, training_config, device)
    joint_fit = fit_operator(hybrid, training_pairs, validation_pairs, training_config, device)
    candidate_name = f'hybrid_r{rank}_m{width}'
    append_candidate(
        candidate_name, 'hybrid', budget['tier'],
        joint_fit.module, joint_fit.history,
    )
    training_histories[candidate_name] = {
        'residual_initialization': [asdict(epoch) for epoch in residual_fit.history],
        'joint_fine_tuning': [asdict(epoch) for epoch in joint_fit.history],
    }
pd.DataFrame(results).query("family == 'hybrid'")

In [ ]:
results_df = pd.DataFrame(results)
candidates = results_df.query("family != 'teacher'").copy().sort_values(['cost_elements', 'delta_loss'])
best_so_far = float('inf')
pareto = []
for _, row in candidates.iterrows():
    is_pareto = row['delta_loss'] < best_so_far
    pareto.append(is_pareto)
    best_so_far = min(best_so_far, row['delta_loss'])
candidates['pareto'] = pareto
results_df = results_df.merge(candidates[['name', 'pareto']], on='name', how='left')
results_df['pareto'] = results_df['pareto'].fillna(False)
results_df.sort_values(['cost_elements', 'delta_loss'])

In [ ]:
plot_df = results_df.query("family != 'teacher'")
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
sns.scatterplot(data=plot_df, x='cost_elements', y='local_relative_mse', hue='family', style='pareto', s=100, ax=axes[0])
sns.scatterplot(data=plot_df, x='cost_elements', y='delta_perplexity', hue='family', style='pareto', s=100, ax=axes[1], legend=False)
sns.scatterplot(data=plot_df, x='local_relative_mse', y='delta_loss', hue='family', style='pareto', s=100, ax=axes[2], legend=False)
axes[0].set(title='Local approximation frontier', xscale='symlog')
axes[1].set(title='Integrated degradation frontier', xscale='symlog')
axes[2].set(title='Local error versus model degradation')
plt.tight_layout()

In [ ]:
artifact = {
    'schema_version': 1,
    'status': 'exploratory',
    'model': {
        'id': model_config.model_id,
        'resolved_revision': getattr(model.config, '_commit_hash', None),
        'target_layer': TARGET_LAYER, 'mlp_path': block.path,
        'requested_config': asdict(model_config),
    },
    'data': {
        'sequence_length': data_config.sequence_length,
        'calibration_batches': data_config.num_calibration_batches,
        'operator_validation_batches': data_config.num_operator_validation_batches,
        'model_validation_batches': data_config.num_model_validation_batches,
        'seed': SEED,
        'requested_config': asdict(data_config),
    },
    'environment': environment_record(),
    'recovery': False, 'ridge': RIDGE,
    'operator_training': asdict(training_config),
    'budgets': list(BUDGETS),
    'baseline_language_model': asdict(baseline_lm),
    'results': results_df.to_dict(orient='records'),
    'training_histories': training_histories,
}
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.write_text(json.dumps(json_value(artifact), indent=2, allow_nan=False), encoding='utf-8')
print(f'Saved cumulative baseline results to {OUTPUT_PATH}')

## Interpretation boundary

The notebook validates a comparison method on one layer and one training seed. A more complex operator adds value only when it improves an equal-cost held-out point or the Pareto frontier. These exploratory results are not evidence that the same family wins at other depths or after multiple simultaneous replacements.